In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import io # Input/Output Module
import os # OS interfaces
import cv2 # OpenCV package
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from urllib import request # module for opening HTTP requests
from matplotlib import pyplot as plt # Plotting library

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA



<div style="width:100%; height:140px">
    <img src="https://www.kuleuven.be/internationaal/thinktank/fotos-en-logos/ku-leuven-logo.png/image_preview" width = auto, heigh = 200px align=left>
</div>


KUL H02A5a Computer Vision: Group Assignment 1
---------------------------------------------------------------
<div style="height:50px"></div>

<span style="color:red">**TODO**: Add your names below.</span>

<span style="color:red">**TODO**: Change the notebook name by replacing the X with your actual group number.</span>

Student names: <span style="color:red">name1, name2, ...</span>.

The goal of this assignment is to explore more advanced techniques for constructing features that better describe objects of interest and to perform face recognition using these features. This assignment will be delivered in groups of 5 (either composed by you or randomly assigned by your TA's).

---------------------------------------------------------------
This notebook is structured as follows:

0. Data loading & Preprocessing
1. Feature Representations
2. Evaluation Metrics 
3. Classifiers
4. Experiments
5. Publishing best results
6. Discussion

Make sure that your notebook is **self-contained** and **fully documented**. Provide strong arguments for the design choices that you made and what insights you got from your experiments. Make use of the *Group assignment 1* forum/discussion board on Toledo if you have any questions.

Good luck! 


<div class="alert alert-block alert-info">
<b>NOTE:</b> This notebook is just a example/template, feel free to adjust in any way you please! Just keep things organised and document accordingly!
</div>

<div class="alert alert-block alert-info">
<b>NOTE:</b> Clearly indicate the improvements that you make!!! You can for instance use titles like: <i>3.1. Improvement: Non-linear SVM with RBF Kernel.<i>
</div>
    
---------------------------------------------------------------
# 0. Data loading & Preprocessing

## 0.1. Loading data
The training set is many times smaller than the test set and this might strike you as odd, however, this is close to a real world scenario where your system might be put through daily use! In this session we will try to do the best we can with the data that we've got! 

In [2]:
# Input data files are available in the read-only "../input/" directory

train = pd.read_csv(
    '/kaggle/input/kul-computer-vision-ga-1-2025/train_set.csv', index_col = 0)
train.index = train.index.rename('id')

test = pd.read_csv(
    '/kaggle/input/kul-computer-vision-ga-1-2025/test_set.csv', index_col = 0)
test.index = test.index.rename('id')

# read the images as numpy arrays and store in "img" column
train['img'] = [cv2.cvtColor(np.load('/kaggle/input/kul-computer-vision-ga-1-2025/train/train_{}.npy'.format(index), allow_pickle=False), cv2.COLOR_BGR2RGB) 
                for index, row in train.iterrows()]

test['img'] = [cv2.cvtColor(np.load('/kaggle/input/kul-computer-vision-ga-1-2025/test/test_{}.npy'.format(index), allow_pickle=False), cv2.COLOR_BGR2RGB) 
                for index, row in test.iterrows()]
  

train_size, test_size = len(train),len(test)

"The training set contains {} examples, the test set contains {} examples.".format(train_size, test_size)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/kul-computer-vision-ga-1-2025/train_set.csv'

*Note: this dataset is a subset of the* [*VGG face dataset*](https://www.robots.ox.ac.uk/~vgg/data/vgg_face/).

## 0.2. A first look
Let's have a look at the data columns and class distribution.

In [ ]:
# The training set contains an identifier, name, image information and class label
train.head(1)

In [ ]:
# The test set only contains an identifier and corresponding image information.

test.head(1)

In [ ]:
# The class distribution in the training set:
train.groupby('name').agg({'img':'count', 'class': 'max'})

Note that **Jesse is assigned the classification label 1**, and **Mila is assigned the classification label 2**. The dataset also contains 20 images of **look alikes (assigned classification label 0)** and the raw images. 

## 0.3. Preprocess data
### 0.3.1 Example: HAAR face detector
In this example we use the [HAAR feature based cascade classifiers](https://opencv-python-tutroals.readthedocs.io/en/latest/py_tutorials/py_objdetect/py_face_detection/py_face_detection.html) to detect faces, then the faces are resized so that they all have the same shape. If there are multiple faces in an image, we only take the first one. 

<div class="alert alert-block alert-info"> <b>NOTE:</b> You can write temporary files to <code>/kaggle/temp/</code> or <code>../../tmp</code>, but they won't be saved outside of the current session
</div>


In [ ]:
class HAARPreprocessor():
    """Preprocessing pipeline built around HAAR feature based cascade classifiers. """
    
    def __init__(self, path, face_size):
        self.face_size = face_size
        file_path = os.path.join(path, "haarcascade_frontalface_default.xml")
        if not os.path.exists(file_path): 
            if not os.path.exists(path):
                os.mkdir(path)
            self.download_model(file_path)
        
        self.classifier = cv2.CascadeClassifier(file_path)
  
    def download_model(self, path):
        url = "https://raw.githubusercontent.com/opencv/opencv/master/data/"\
            "haarcascades/haarcascade_frontalface_default.xml"
        
        with request.urlopen(url) as r, open(path, 'wb') as f:
            f.write(r.read())
            
    def detect_faces(self, img):
        """Detect all faces in an image."""
        
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return self.classifier.detectMultiScale(
            img_gray,
            scaleFactor=1.2,
            minNeighbors=5,
            minSize=(30, 30),
            flags=cv2.CASCADE_SCALE_IMAGE
        )
        
    def extract_faces(self, img):
        """Returns all faces (cropped) in an image."""
        
        faces = self.detect_faces(img)

        return [img[y:y+h, x:x+w] for (x, y, w, h) in faces]
    
    def preprocess(self, data_row):
        faces = self.extract_faces(data_row['img'])
        
        # if no faces were found, return None
        if len(faces) == 0:
            nan_img = np.empty(self.face_size + (3,))
            nan_img[:] = np.nan
            return nan_img
        
        # only return the first face
        return cv2.resize(faces[0], self.face_size, interpolation = cv2.INTER_AREA)
            
    def __call__(self, data):
        return np.stack([self.preprocess(row) for _, row in data.iterrows()]).astype(int)

**Visualise**

Let's plot a few examples.

In [ ]:
# parameter to play with 
FACE_SIZE = (100, 100)

def plot_image_sequence(data, n, imgs_per_row=7):
    n_rows = 1 + int(n/(imgs_per_row+1))
    n_cols = min(imgs_per_row, n)

    f,ax = plt.subplots(n_rows,n_cols, figsize=(10*n_cols,10*n_rows))
    for i in range(n):
        if n == 1:
            ax.imshow(data[i])
        elif n_rows > 1:
            ax[int(i/imgs_per_row),int(i%imgs_per_row)].imshow(data[i])
        else:
            ax[int(i%n)].imshow(data[i])
    plt.show()

    
#preprocessed data 
preprocessor = HAARPreprocessor(path = '../../tmp', face_size=FACE_SIZE)

train_X, train_y = preprocessor(train), train['class'].values
test_X = preprocessor(test)



In [ ]:
# plot faces of Michael and Sarah

plot_image_sequence(train_X[train_y == 0], n=20, imgs_per_row=10)

In [ ]:
# plot faces of Jesse

plot_image_sequence(train_X[train_y == 1], n=30, imgs_per_row=10)

In [ ]:
# plot faces of Mila

plot_image_sequence(train_X[train_y == 2], n=30, imgs_per_row=10)

## 0.4. Store Preprocessed data (optional)
<div class="alert alert-block alert-info">
<b>NOTE:</b> You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All". Feel free to use this to store intermediary results.
</div>

In [ ]:
# save preprocessed data
# prep_path = '/kaggle/working/prepped_data/'
# if not os.path.exists(prep_path):
#     os.mkdir(prep_path)
    
# np.save(os.path.join(prep_path, 'train_X.npy'), train_X)
# np.save(os.path.join(prep_path, 'train_y.npy'), train_y)
# np.save(os.path.join(prep_path, 'test_X.npy'), test_X)

# load preprocessed data
# prep_path = '/kaggle/working/prepped_data/'
# if not os.path.exists(prep_path):
#     os.mkdir(prep_path)
# train_X = np.load(os.path.join(prep_path, 'train_X.npy'))
# train_y = np.load(os.path.join(prep_path, 'train_y.npy'))
# test_X = np.load(os.path.join(prep_path, 'test_X.npy'))

Now we are ready to rock!

# 1. Feature Representations
## 1.0. Example: Identify feature extractor
Our example feature extractor doesn't actually do anything... It just returns the input:
$$
\forall x : f(x) = x.
$$

It does make for a good placeholder and baseclass ;).

In [4]:
class IdentityFeatureExtractor:
    """A simple function that returns the input"""
    
    def transform(self, X):
        return X
    
    def __call__(self, X):
        return self.transform(X)

## 1.1. Baseline 1: HOG feature extractor/Scale Invariant Feature Transform
...

In [ ]:
class HOGFeatureExtractor(IdentityFeatureExtractor):
    """TODO: this feature extractor is under construction"""
    
    def __init__(**params):
        self.params = params
        
    def transform(self, X):
        raise NotImplmentedError

In [5]:
class SIFTFeatureExtractor(IdentityFeatureExtractor):
    """
    SIFT feature extractor for face images.
    Uses a dense grid of keypoints and aggregates descriptors
    with mean and std to obtain a fixed-length vector per image.
    """
    def __init__(self, image_size=(64, 64), step=8, kp_size=8, use_clahe=True):
        self.image_size = image_size
        self.step = step
        self.kp_size = kp_size
        self.use_clahe = use_clahe
        self.sift = cv2.SIFT_create()
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)) if use_clahe else None

    def _prepare_image(self, img):
        img = np.asarray(img)
        img = np.nan_to_num(img, nan=0.0)

        if img.dtype != np.uint8:
            img = img.astype(np.float32)
            if img.max() <= 1.0:
                img = img * 255.0
            img = np.clip(img, 0, 255).astype(np.uint8)

        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        img = cv2.resize(img, self.image_size, interpolation=cv2.INTER_AREA)

        if self.clahe is not None:
            img = self.clahe.apply(img)

        return img

    def _make_keypoints(self):
        width, height = self.image_size
        keypoints = [
            cv2.KeyPoint(float(x), float(y), self.kp_size)
            for y in range(self.step // 2, height, self.step)
            for x in range(self.step // 2, width, self.step)
        ]
        return keypoints

    def transform(self, X):
        keypoints = self._make_keypoints()
        features = []

        for img in X:
            img = self._prepare_image(img)
            _, descriptors = self.sift.compute(img, keypoints)

            if descriptors is None or len(descriptors) == 0:
                feat = np.zeros(256, dtype=np.float32)  # 128 mean + 128 std
            else:
                mean_desc = descriptors.mean(axis=0)
                std_desc = descriptors.std(axis=0)
                feat = np.concatenate([mean_desc, std_desc]).astype(np.float32)

            features.append(feat)

        return np.vstack(features)

    def __call__(self, X):
        return self.transform(X)


The implemented `SIFTFeatureExtractor` computes a fixed-length SIFT-based representation for each image.  
Although the class is named `SIFTFeatureExtractor`, the implementation uses a **dense SIFT strategy** instead of relying only on automatically detected sparse keypoints. This choice is better suited for face images, where important visual information is distributed over consistent regions such as the eyes, nose, mouth, and facial contour.

The extraction pipeline is the following:

1. **Image preprocessing**
   - Missing values are replaced with zeros.
   - Images are converted to `uint8` format.
   - RGB images are converted to grayscale.
   - All images are resized to a fixed resolution of `64 x 64` pixels.
   - CLAHE (Contrast Limited Adaptive Histogram Equalization) is applied to improve local contrast and make the descriptor more robust to illumination changes.

2. **Dense keypoint generation**
   - Instead of depending on the keypoints detected automatically by SIFT, a regular grid of keypoints is defined over the whole image.
   - This ensures that descriptors are extracted from similar facial regions across all samples.

3. **SIFT descriptor computation**
   - SIFT descriptors are computed at the grid locations.
   - Each descriptor has 128 dimensions.

4. **Feature aggregation**
   - Since the number of local descriptors per image can vary, the descriptors are aggregated into a fixed-length feature vector.
   - For each image, the **mean** and **standard deviation** of the SIFT descriptors are computed and concatenated.
   - This produces a final feature vector of **256 dimensions** (`128 mean + 128 standard deviation`) for each image.


### 1.1.1. t-SNE Plots
...

#### 1.1.1.2 SIFT t-SNE Plots

In [ ]:
# 1) Extract SIFT features
sift_extractor = SIFTFeatureExtractor(
    image_size=(64, 64),
    step=4,
    kp_size=6,
    use_clahe=True
)

X_sift = sift_extractor.transform(train_X)

# 2) Safety step
X_sift = np.nan_to_num(X_sift, nan=0.0)

# 3) Scale features
X_scaled = StandardScaler().fit_transform(X_sift)

# 4) PCA before t-SNE
n_pca = min(20, X_scaled.shape[0] - 1, X_scaled.shape[1])
X_pca = PCA(n_components=n_pca, random_state=42).fit_transform(X_scaled)

# 5) t-SNE
perplexity = max(3, min(10, len(X_pca) // 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    random_state=42
)

X_tsne = tsne.fit_transform(X_pca)

# 6) Plot
y_train_array = np.asarray(train_y)

plt.figure(figsize=(8, 6))

for label in np.unique(y_train_array):
    idx = (y_train_array == label)
    plt.scatter(X_tsne[idx, 0], X_tsne[idx, 1], s=25, alpha=0.8, label=str(label))

plt.title("t-SNE of SIFT features")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

##### 1.1.1.2.1 Choice of Parameters

The parameters of the SIFT extractor were selected to obtain a stable and informative representation for face images.

- **Image size = 64 x 64**  
  All images were resized to a common resolution in order to standardize the extraction process.  
  A size of `64 x 64` is large enough to preserve the main facial structures while keeping the computational cost low.

- **Dense grid step = 4**  
  A smaller step leads to a denser sampling of the image and therefore captures more local facial details.  
  This is useful for faces, since relevant information is spread across several regions and not only around a few strong keypoints.

- **Keypoint size = 6**  
  This parameter defines the local neighborhood from which each SIFT descriptor is computed.  
  A moderate keypoint size was chosen to balance fine local detail and robustness to small variations in appearance.

- **CLAHE = True**  
  CLAHE was applied to improve local contrast and reduce the impact of illumination differences.  
  This preprocessing step is especially useful in facial datasets, where lighting conditions can strongly affect handcrafted descriptors.

- **Mean + standard deviation aggregation**  
  Instead of concatenating all local descriptors directly, the descriptors were summarized using their mean and standard deviation.  
  This provides a compact fixed-length representation and makes the features more suitable for visualization and further analysis.

Overall, these parameters were chosen to make the descriptor more stable, more comparable across images, and more adapted to the structure of facial data.

### 1.1.2. Discussion


#### 1.1.2.1 Discussion of the results using SIFT
The t-SNE projection of the SIFT features shows a partial but meaningful separation between the three classes.

- **Class 1** is mostly concentrated in the lower part of the embedding.
- **Class 0** tends to appear more in the upper-left / central region
- **Class 2** is more visible in the upper-right region.  

This indicates that the SIFT-based representation captures relevant visual patterns that help distinguish the classes.

The separation is not perfect, since some overlap is still present between the groups. However, the overall structure of the embedding suggests that the extracted SIFT features are moderately discriminative for this task.

A complete separation is not necessarily expected, especially for face images, because different classes may still share similar local structures, facial layouts, or illumination conditions. In addition, t-SNE is only a two-dimensional projection of a higher-dimensional feature space, so some information loss is unavoidable.

## 1.2. Baseline 2: PCA feature extractor
...

In [ ]:
class PCAFeatureExtractor(IdentityFeatureExtractor):
    """TODO: this feature extractor is under construction"""
    
    def __init__(self, n_components):
        self.n_components = n_components
        
    def transform(self, X):
        raise NotImplmentedError
        
    def inverse_transform(self, X):
        raise NotImplmentedError

### 1.2.1. Eigenface Plots
...

### 1.2.2. Feature Space Plots
...

### 1.2.3. Discussion
...

# 2. Evaluation Metrics
## 2.0. Example: Accuracy
As example metric we take the accuracy. Informally, accuracy is the proportion of correct predictions over the total amount of predictions. It is used a lot in classification but it certainly has its disadvantages...

In [ ]:
from sklearn.metrics import accuracy_score

# 3. Classifiers
## 3.0. Example: The *'not so smart'* classifier
This random classifier is not very complicated. It makes predictions at random, based on the distribution obseved in the training set. **It thus assumes** that the class labels of the test set will be distributed similarly to the training set.

In [ ]:
class RandomClassificationModel:
    """Random classifier, draws a random sample based on class distribution observed 
    during training."""
    
    def fit(self, X, y):
        """Adjusts the class ratio instance variable to the one observed in y. 

        Parameters
        ----------
        X : tensor
            Training set
        y : array
            Training set labels

        Returns
        -------
        self : RandomClassificationModel
        """
        
        self.classes, self.class_ratio = np.unique(y, return_counts=True)
        self.class_ratio = self.class_ratio / self.class_ratio.sum()
        return self
        
    def predict(self, X):
        """Samples labels for the input data. 

        Parameters
        ----------
        X : tensor
            dataset
            
        Returns
        -------
        y_star : array
            'Predicted' labels
        """

        np.random.seed(0)
        return np.random.choice(self.classes, size = X.shape[0], p=self.class_ratio)
    
    def __call__(self, X):
        return self.predict(X)
    

## 3.1. Baseline 1: My favorite classifier
...

In [ ]:
class FavoriteClassificationModel:
    """TODO: this classifier is under construction."""
    
    def fit(self, X, y):
        raise NotImplmentedError
        
    def predict(self, X):
        raise NotImplmentedError

# 4. Experiments
<div class="alert alert-block alert-info"> <b>NOTE:</b> Do <i>NOT</i> use this section to keep track of every little change you make in your code! Instead, highlight the most important findings and the major (best) pipelines that you've discovered.  
</div>
<br>

## 4.0. Example: basic pipeline
The basic pipeline takes any input and samples a label based on the class label distribution of the training set. As expected the performance is very poor, predicting approximately 1/4 correctly on the training set. There is a lot of room for improvement but this is left to you ;). 

In [ ]:
feature_extractor = IdentityFeatureExtractor() 
classifier = RandomClassificationModel()

# train the model on the features
classifier.fit(feature_extractor(train_X), train_y)

# model/final pipeline
model = lambda X: classifier(feature_extractor(X))

In [ ]:
# evaluate performance of the model on the training set
train_y_star = model(train_X)

"The performance on the training set is {:.2f}. This however, does not tell us much about the actual performance (generalisability).".format(
    accuracy_score(train_y, train_y_star))

In [ ]:
# predict the labels for the test set 
test_y_star = model(test_X)

# 5. Publishing best results

In [ ]:
submission = test.copy().drop('img', axis = 1)
submission['class'] = test_y_star

submission

In [ ]:
submission.to_csv('submission.csv')

# 6. Discussion
...

In summary we contributed the following: 
* 
